## 1. Load Data and Baseline GNN Predictions

Load the pre-generated baseline predictions from the GCN models.


In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

# Choose dataset: 'BBBP', 'BACE', or 'HIV'
DATASET = 'BBBP'
K = 4 # number of neighbors to use for LLM boosting
LOCAL_LLM = "mistral:7b-instruct"
SPLIT = "val" # val or test
PROMPT = "V3"
KNN = True
PARAM = [0.3, 0.9] # temperature, top_p

# Load the predictions from CSV files
test_val_df = pd.read_csv(f'baseline_predictions/{DATASET}_{SPLIT}_predictions.csv')
train_df = pd.read_csv(f'baseline_predictions/{DATASET}_train_data.csv')

# Extract the data
test_val_smiles = test_val_df['SMILES'].tolist()
test_val_labels = test_val_df['True_Label'].to_numpy()
y_pred_gnn_initial = test_val_df['GNN_Prediction'].to_numpy()

train_smiles = train_df['SMILES'].tolist()
train_labels = train_df['Label'].tolist()

print(f"Dataset: {DATASET}")
print(f"Test set size: {len(test_val_smiles)}")
print(f"Train set size: {len(train_smiles)}")
print(f"GNN Test AUC: {roc_auc_score(test_val_labels, y_pred_gnn_initial):.4f}")

Dataset: BBBP
Test set size: 204
Train set size: 1631
GNN Test AUC: 0.9128


## 2. Find K-Most Similar Molecules

Use Tanimoto similarity on Morgan fingerprints to find the K most similar training molecules for each test molecule.


In [2]:
from typing import List, Literal, Optional, Iterable, Dict, Any
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, RDKFingerprint
from sklearn.metrics import roc_auc_score

def smiles_to_fingerprint(
    smiles: str,
    kind: Literal["morgan", "rdk"] = "morgan",
    radius: int = 2,
    nBits: int = 2048
):
    """Convert SMILES to RDKit fingerprint, return None if fails"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    if kind == "morgan":
        return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits)
    elif kind == "rdk":
        return RDKFingerprint(mol, fpSize=nBits)
    else:
        raise ValueError("Unknown fingerprint kind")

def batch_smiles_to_fps(
    smiles_list: Iterable[str],
    kind: Literal["morgan", "rdk"] = "morgan",
    radius: int = 2,
    nBits: int = 2048
):
    return [smiles_to_fingerprint(s, kind=kind, radius=radius, nBits=nBits) for s in smiles_list]

def find_k_most_similar_molecules(
    test_smiles: Iterable[str],
    train_smiles: Iterable[str],
    y_train: Iterable[Any],
    k_neighbors: int = 5,
    fp_kind: Literal["morgan", "rdk"] = "morgan",
    fp_radius: int = 2,
    fp_nBits: int = 2048,
    return_indices: bool = True
) -> List[Dict[str, Any]]:
    """
    For each test molecule, find the K most similar molecules in the training set based on Tanimoto similarity.
    """
    # Precompute fingerprints of the training set
    train_smiles = list(train_smiles)
    y_train = np.asarray(list(y_train))
    train_fps = batch_smiles_to_fps(train_smiles, kind=fp_kind, radius=fp_radius, nBits=fp_nBits)

    # Record which training samples are valid
    valid_train = np.array([fp is not None for fp in train_fps])
    if not valid_train.any():
        raise ValueError("No valid fingerprints in training set.")
    valid_train_fps = [fp for fp in train_fps if fp is not None]
    valid_train_smiles = [s for s, v in zip(train_smiles, valid_train) if v]
    valid_y_train = y_train[valid_train]

    results = []

    # Iterate through test set
    for q_idx, q_smiles in enumerate(test_smiles):
        q_fp = smiles_to_fingerprint(q_smiles, kind=fp_kind, radius=fp_radius, nBits=fp_nBits)
        if q_fp is None:
            results.append({
                "test_index": q_idx,
                "test_smiles": q_smiles,
                "neighbors_smiles": [],
                "neighbors_labels": [],
                "neighbors_similarity": [],
                "neighbors_indices_in_train": [] if return_indices else None,
                "note": "invalid test SMILES"
            })
            continue

        sims = np.array(DataStructs.BulkTanimotoSimilarity(q_fp, valid_train_fps), dtype=float)

        # Select top K
        k = min(k_neighbors, sims.size)
        topk_idx = np.argpartition(sims, -k)[-k:]
        topk_idx = topk_idx[np.argsort(sims[topk_idx])[::-1]]

        results.append({
            "test_index": q_idx,
            "test_smiles": q_smiles,
            "neighbors_smiles": [valid_train_smiles[i] for i in topk_idx],
            "neighbors_labels":  [valid_y_train[i]    for i in topk_idx],
            "neighbors_similarity": sims[topk_idx].tolist(),
            "neighbors_indices_in_train": (
                np.nonzero(valid_train)[0][topk_idx].tolist() if return_indices else None
            )
        })

    return results


In [3]:
results = find_k_most_similar_molecules(
    test_smiles=test_val_smiles,
    train_smiles=train_smiles,
    y_train=train_labels,
    k_neighbors=K,
    fp_kind="morgan",
    fp_radius=2,
    fp_nBits=2048,
)

# Convert to neighbor list format
def to_neighbor_list(res_i):
    return [
        {
            "smiles": s,
            "y_true": float(y) if str(y).replace('.','',1).isdigit() else y,
            "tanimoto": float(sim)
        }
        for s, y, sim in zip(
            res_i["neighbors_smiles"],
            res_i["neighbors_labels"],
            res_i["neighbors_similarity"]
        )
    ]

similar_mols = [to_neighbor_list(r) for r in results]
print(f"Found {K} similar molecules for each of {len(test_val_smiles)} test molecules")


[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerat

Found 4 similar molecules for each of 204 test molecules


[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerator
[17:37:20] DEPRECATION WARNING: please use MorganGenerat

## 3. Setup Ollama Client

Initialize the local LLM client using Ollama. Make sure Ollama is running before executing this cell!


In [4]:
import time
import random
import re
from typing import Optional
import requests
import json

# Ollama configuration
OLLAMA_MODEL = LOCAL_LLM # or "mistral", "llama2", etc. - adjust based on what you have installed
OLLAMA_BASE_URL = "http://localhost:11434"  # Default Ollama API endpoint

# Prompt builder
def prompt_for_llm(current_molecule_smiles, initial_pred_gnn, similar_historical_molecules):
    return f"""
[INST]You are assisting with molecular property prediction.

Goal: refine a baseline prediction for BBB penetration using both
(a) chemical reasoning from structure and
(b) information from K similar labeled molecules.

Steps:
1) Analyze the SMILES for key chemical features 
   (e.g., polarity, hydrogen bond donors/acceptors, aromatic rings, molecular size, lipophilicity).
2) Recall general rules:
   - Polar or charged groups lower BBB permeability.
   - Lipophilic, compact, neutral molecules cross more easily.
3) Compare structural motifs with the K similar molecules to see which patterns correlate with their outcomes.
   - y=1 indicates BBB penetration and y=0 indicates no penetration.
   - Tanimoto similarity between 0 and 1. 1 indicates identical molecules, 0 indicates completely different molecules. 
4) Combine the initial predicted probability with chemical reasoning and these examples.
   Keep adjustments moderate unless strong evidence supports change.

Output format:
Return **only** a valid JSON object, with no text before or after it, following this schema:

{{
    "explanation": "<one short sentence describing the main chemical reason for the adjustment>",
    "probability": <float between 0 and 1>
}}

Input:
- Current molecule (SMILES): {current_molecule_smiles}
- Initial prediction (probability): {initial_pred_gnn}
- K similar molecules (SMILES | label y | similarity):
  {similar_historical_molecules}
[/INST]

""".strip()

# Numeric parser
_number_re = re.compile(r"[-+]?(?:\d+\.\d+|\d+)(?:[eE][-+]?\d+)?")
def parse_float(s: str, initial_pred: float = None) -> float:
    """
    Parse float from LLM response. 
    First tries to parse as JSON and extract 'probability' field.
    Falls back to regex if JSON parsing fails.
    """
    if not s or not s.strip():
        raise ValueError(f"Empty response string")
    
    # Try JSON parsing first (expected format)
    try:
        # Try to find JSON object in the response
        # Look for {...} pattern
        json_match = re.search(r'\{[^{}]*"probability"[^{}]*\}', s, re.DOTALL)
        if json_match:
            json_str = json_match.group(0)
            parsed = json.loads(json_str)
            if "probability" in parsed:
                prob = float(parsed["probability"])
                # Check if it's different from initial prediction if provided
                if initial_pred is not None and abs(prob - initial_pred) < 0.001:
                    raise ValueError(f"JSON probability ({prob:.6f}) matches initial prediction ({initial_pred:.6f})")
                return prob
        
        # Try parsing the whole string as JSON
        parsed = json.loads(s.strip())
        if "probability" in parsed:
            prob = float(parsed["probability"])
            if initial_pred is not None and abs(prob - initial_pred) < 0.001:
                raise ValueError(f"JSON probability ({prob:.6f}) matches initial prediction ({initial_pred:.6f})")
            return prob
    except (json.JSONDecodeError, KeyError, ValueError) as e:
        # If JSON parsing fails, fall back to regex
        pass
    
    # Fallback to regex-based parsing
    all_numbers = _number_re.findall(s)
    if not all_numbers:
        raise ValueError(f"Could not find a numeric answer in: {s!r}")
    
    # If we have an initial prediction, try to find a number that's different
    if initial_pred is not None:
        for num_str in all_numbers:
            num_val = float(num_str)
            # If the number is significantly different from initial, use it
            if abs(num_val - initial_pred) > 0.001:
                return num_val
        # If all numbers are the same as initial, raise an error
        raise ValueError(f"All numbers in response match initial prediction {initial_pred}: {s!r}")
    
    # Otherwise, just return the first number
    return float(all_numbers[0])

# Ollama client
class _OllamaClient:
    def __init__(self, model: str = OLLAMA_MODEL, base_url: str = OLLAMA_BASE_URL):
        self.model = model
        self.base_url = base_url
        self.api_url = f"{base_url}/api/generate"
        
        # Test connection
        try:
            response = requests.get(f"{base_url}/api/tags", timeout=5)
            if response.status_code != 200:
                raise ConnectionError(f"Cannot connect to Ollama at {base_url}")
        except requests.exceptions.RequestException as e:
            raise ConnectionError(
                f"Cannot connect to Ollama at {base_url}. "
                f"Make sure Ollama is running (ollama serve). Error: {e}"
            )

    def generate(self, prompt: str, system_prompt: Optional[str] = None) -> str:
        """Generate a response from Ollama."""
        payload = {
            "model": self.model,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": PARAM[0],
                "top_p": PARAM[1]
            }
        }
        
        if system_prompt:
            payload["system"] = system_prompt
        
        response = requests.post(
            self.api_url,
            json=payload,
            timeout=240
        )
        
        if response.status_code != 200:
            raise RuntimeError(f"Ollama API error: {response.status_code} - {response.text}")
        
        result = response.json()
        if "response" not in result:
            raise RuntimeError(f"Unexpected Ollama response format: {result}")
        
        return result["response"].strip()

def build_ollama_client(model: str = OLLAMA_MODEL, base_url: str = OLLAMA_BASE_URL):
    """Build an Ollama client."""
    return _OllamaClient(model=model, base_url=base_url)

# Retry wrapper
def call_llm_with_retry(client, prompt, max_attempts=3, base_sleep=1.0, max_sleep=10.0, **_):
    """Call LLM with retry logic."""
    system_prompt = "Return only a single number (float). No text, no units."
    attempt = 0
    
    while True:
        attempt += 1
        try:
            text = client.generate(prompt, system_prompt=system_prompt)
            if not text.strip():
                raise RuntimeError("Empty response from model")
            return text.strip()
        except Exception as e:
            msg = str(e).lower()
            retriable = any(x in msg for x in [
                "timeout", "temporarily", "unavailable", "connection reset",
                "read timed out", "overloaded", "connection error"
            ])
            
            if not retriable or attempt >= max_attempts:
                raise
            
            sleep_s = min(max_sleep, base_sleep * (2 ** (attempt - 1)))
            sleep_s *= random.uniform(0.7, 1.3)
            print(f"  Retrying after {sleep_s:.1f}s (attempt {attempt}/{max_attempts})...")
            time.sleep(sleep_s)

# Initialize client
print(f"Initializing Ollama client with model: {OLLAMA_MODEL}")
print(f"Make sure Ollama is running: 'ollama serve'")
print(f"And the model is downloaded: 'ollama pull {OLLAMA_MODEL}'")
client = build_ollama_client(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL)
print("✓ Ollama client initialized successfully!")


Initializing Ollama client with model: mistral:7b-instruct
Make sure Ollama is running: 'ollama serve'
And the model is downloaded: 'ollama pull mistral:7b-instruct'
✓ Ollama client initialized successfully!


## 4. Run LLM Boosting

Process each test molecule through the LLM to refine the GNN predictions.

**Note**: Start with `num_samples = 10` to test, then increase to the full dataset.


In [ ]:
LAMBDA = 0.6
def format_neighbors(neighbor_list):
    """Format neighbor list for the prompt."""
    if not neighbor_list:
        return "None"
    lines = []
    for i, nb in enumerate(neighbor_list, 1):
        lines.append(f"{i}. SMILES: {nb['smiles']} | y: {float(nb['y_true']):.6g} | Tanimoto: {float(nb['tanimoto']):.3f}")
    return "\n".join(lines)

# Collect results
results_list = []

# Main loop - you can run on a subset first to test
# num_samples = 10  # Start with 10 samples to test
num_samples = len(test_val_smiles)  # Uncomment to run on all samples

print(f"\n{'='*60}")
print(f"Starting LLM boosting with {num_samples} samples")
print(f"{'='*60}\n")

for i in range(num_samples):
    print(f"\n=== Processing sample {i+1}/{num_samples} ===")
    current_smiles = test_val_smiles[i]
    initial_pred_gnn = float(y_pred_gnn_initial[i])
    k_neighbors_block = format_neighbors(similar_mols[i])
    
    # KNN model
    initial_pred_gnn = float(y_pred_gnn_initial[i])
    if KNN:
        similarities = [similar_mols[i][j]['tanimoto'] for j in range(K)]
        if sum(similarities) > 0:
            similarities = [s/sum(similarities) for s in similarities]
        property = [similar_mols[i][j]['y_true'] for j in range(K)]
        initial_pred_gnn = LAMBDA*initial_pred_gnn + (1 - LAMBDA) * sum([s*p for s,p in zip(similarities, property)])

    prompt = prompt_for_llm(current_smiles, initial_pred_gnn, k_neighbors_block)
    print(prompt)

    try:
        raw = call_llm_with_retry(client, prompt)
        print(raw)
        y_llm = parse_float(raw)
        print(f"  GNN: {initial_pred_gnn:.4f}, LLM: {y_llm:.4f}, True: {test_val_labels[i]:.0f}")
    except Exception as e:
        print(f"  ✗ LLM call failed at index {i}: {e}")
        continue

    results_list.append([y_llm, initial_pred_gnn, float(test_val_labels[i])])

# Create DataFrame
dt_final = pd.DataFrame(results_list, columns=["LLM_Prediction", "GNN_Initial", "Actual_Label"])
print(f"\n{'='*60}")
print(f"✓ {len(dt_final)} predictions completed!")
print(f"{'='*60}")



Starting LLM boosting with 204 samples


=== Processing sample 1/204 ===
[INST]You are assisting with molecular property prediction.

Goal: refine a baseline prediction for BBB penetration using both
(a) chemical reasoning from structure and
(b) information from K similar labeled molecules.

Steps:
1) Analyze the SMILES for key chemical features 
   (e.g., polarity, hydrogen bond donors/acceptors, aromatic rings, molecular size, lipophilicity).
2) Recall general rules:
   - Polar or charged groups lower BBB permeability.
   - Lipophilic, compact, neutral molecules cross more easily.
3) Compare structural motifs with the K similar molecules to see which patterns correlate with their outcomes.
   - y=1 indicates BBB penetration and y=0 indicates no penetration.
   - Tanimoto similarity between 0 and 1. 1 indicates identical molecules, 0 indicates completely different molecules. 
4) Combine the initial predicted probability with chemical reasoning and these examples.
   Keep adjustments 

## 5. Evaluate Results

Compare the LLM-boosted predictions against the baseline GNN across multiple metrics.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (roc_auc_score, average_precision_score, log_loss, 
                             f1_score, accuracy_score, matthews_corrcoef, roc_curve)

# Set plot style
sns.set_style("whitegrid")

y_true = pd.to_numeric(dt_final["Actual_Label"], errors="coerce").to_numpy(dtype=float)
y_llm  = pd.to_numeric(dt_final["LLM_Prediction"], errors="coerce").to_numpy(dtype=float)
y_gnn  = pd.to_numeric(dt_final["GNN_Initial"], errors="coerce").to_numpy(dtype=float)

# mask = y_llm < 1
# y_true = y_true[mask]
# y_llm = y_llm[mask]
# y_gnn = y_gnn[mask]

# Remove NaN/inf values and clip to [0,1]
np.clip(y_llm, 0, 1, out=y_llm)
np.clip(y_gnn, 0, 1, out=y_gnn)
mask = np.isfinite(y_true) & np.isfinite(y_llm) & np.isfinite(y_gnn)
y_true, y_llm, y_gnn = y_true[mask], y_llm[mask], y_gnn[mask]

print(f"\n{'='*60}")
print(f"EVALUATION RESULTS - {DATASET} Dataset")
print(f"{'='*60}\n")

# 1) Brier/MSE
mse_llm = np.mean((y_llm - y_true)**2)
mse_gnn = np.mean((y_gnn - y_true)**2)
print(f"Brier/MSE  LLM={mse_llm:.5f}  GNN={mse_gnn:.5f}  Δ={mse_llm-mse_gnn:+.5f}")

# 2) Probability-based metrics
auc_llm = roc_auc_score(y_true, y_llm)
auc_gnn = roc_auc_score(y_true, y_gnn)
ap_llm  = average_precision_score(y_true, y_llm)
ap_gnn  = average_precision_score(y_true, y_gnn)
ll_llm  = log_loss(y_true, np.clip(y_llm,1e-6,1-1e-6))
ll_gnn  = log_loss(y_true, np.clip(y_gnn,1e-6,1-1e-6))

print(f"AUC        LLM={auc_llm:.4f}  GNN={auc_gnn:.4f}  Δ={auc_llm-auc_gnn:+.4f}")
print(f"PR-AUC     LLM={ap_llm:.4f}   GNN={ap_gnn:.4f}   Δ={ap_llm-ap_gnn:+.4f}")
print(f"LogLoss    LLM={ll_llm:.5f}  GNN={ll_gnn:.5f}  Δ={ll_llm-ll_gnn:+.5f}")

# 3) Threshold-based metrics
def optimal_thresh(y_true, y_prob):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    j = tpr - fpr
    return thr[j.argmax()]

t_llm = optimal_thresh(y_true, y_llm)
t_gnn = optimal_thresh(y_true, y_gnn)
yhat_llm = (y_llm >= t_llm).astype(int)
yhat_gnn = (y_gnn >= t_gnn).astype(int)

acc_llm = accuracy_score(y_true, yhat_llm)
acc_gnn = accuracy_score(y_true, yhat_gnn)
f1_llm  = f1_score(y_true, yhat_llm)
f1_gnn  = f1_score(y_true, yhat_gnn)
mcc_llm = matthews_corrcoef(y_true, yhat_llm)
mcc_gnn = matthews_corrcoef(y_true, yhat_gnn)

print(f"ACC        LLM={acc_llm:.4f}  GNN={acc_gnn:.4f}  Δ={acc_llm-acc_gnn:+.4f}")
print(f"F1         LLM={f1_llm:.4f}   GNN={f1_gnn:.4f}   Δ={f1_llm-f1_gnn:+.4f}")
print(f"MCC        LLM={mcc_llm:.4f}  GNN={mcc_gnn:.4f}  Δ={mcc_llm-mcc_gnn:+.4f}")

# 4) Analysis: Improvement statistics
print(f"\n{'='*60}")
print("IMPROVEMENT ANALYSIS")
print(f"{'='*60}\n")

# Calculate per-sample absolute errors
error_llm = np.abs(y_llm - y_true)
error_gnn = np.abs(y_gnn - y_true)

# Count improvements
improved = error_llm < error_gnn
worsened = error_llm > error_gnn
unchanged = error_llm == error_gnn

pct_improved = 100 * improved.sum() / len(y_true)
pct_worsened = 100 * worsened.sum() / len(y_true)
pct_unchanged = 100 * unchanged.sum() / len(y_true)

print(f"Samples where LLM improved prediction:  {improved.sum():4d} / {len(y_true)} ({pct_improved:.1f}%)")
print(f"Samples where LLM worsened prediction:  {worsened.sum():4d} / {len(y_true)} ({pct_worsened:.1f}%)")
print(f"Samples where LLM unchanged:            {unchanged.sum():4d} / {len(y_true)} ({pct_unchanged:.1f}%)")

# Average improvement
avg_error_reduction = np.mean(error_gnn - error_llm)
print(f"\nAverage absolute error reduction: {avg_error_reduction:+.5f}")

print(f"\n{'='*60}")


## 5b. Visualizations

Plot ROC curves and prediction distributions to visualize model performance.


In [ ]:
# Create visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. ROC Curves
fpr_gnn, tpr_gnn, _ = roc_curve(y_true, y_gnn)
fpr_llm, tpr_llm, _ = roc_curve(y_true, y_llm)

axes[0].plot(fpr_gnn, tpr_gnn, 'b-', linewidth=2, label=f'GNN (AUC={auc_gnn:.3f})')
axes[0].plot(fpr_llm, tpr_llm, 'r-', linewidth=2, label=f'LLM-Boosted (AUC={auc_llm:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.3, label='Random')
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title(f'ROC Curves - {DATASET} Dataset', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right', fontsize=10)
axes[0].grid(True, alpha=0.3)

# 2. Distribution of Predictions by True Label
for true_label, label_name, color in [(0, 'Negative (y=0)', 'blue'), (1, 'Positive (y=1)', 'red')]:
    mask_label = y_true == true_label
    
    # GNN predictions
    axes[1].hist(y_gnn[mask_label], bins=20, alpha=0.4, color=color, 
                 label=f'GNN - {label_name} (n={mask_label.sum()})', density=True)
    # LLM predictions
    axes[1].hist(y_llm[mask_label], bins=20, alpha=0.4, color=color, 
                 edgecolor='black', linewidth=1.5, linestyle='--',
                 label=f'LLM - {label_name} (n={mask_label.sum()})', density=True)

axes[1].set_xlabel('Predicted Probability', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Prediction Distributions', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=9, loc='upper center')
axes[1].grid(True, alpha=0.3, axis='y')

# 3. Comparison: GNN vs LLM Predictions
scatter_colors = ['green' if imp else 'orange' if wor else 'gray' 
                  for imp, wor in zip(improved, worsened)]
axes[2].scatter(y_gnn, y_llm, c=scatter_colors, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
axes[2].plot([0, 1], [0, 1], 'k--', linewidth=2, alpha=0.5, label='No change')
axes[2].set_xlabel('GNN Prediction', fontsize=12)
axes[2].set_ylabel('LLM Prediction', fontsize=12)
axes[2].set_title('GNN vs LLM Predictions', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3)

# Custom legend for scatter plot
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', alpha=0.6, label=f'Improved ({improved.sum()})'),
    Patch(facecolor='orange', alpha=0.6, label=f'Worsened ({worsened.sum()})'),
    Patch(facecolor='gray', alpha=0.6, label=f'Unchanged ({unchanged.sum()})')
]
axes[2].legend(handles=legend_elements, loc='upper left', fontsize=10)

plt.tight_layout()
plt.savefig(f'llm_gnn_evaluation_{DATASET}_local.png', dpi=300, bbox_inches='tight')
plt.show()


## 6. Save Results

Save the predictions and summary statistics to CSV files.


In [ ]:
import os
from datetime import datetime

# Generate unique run identifier
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

# Extract temperature and top_p from PARAM
temperature, top_p = PARAM[0], PARAM[1]

# Save detailed results as a single row to shared CSV
detailed_results = pd.DataFrame({
    'run_id': [run_id],
    'dataset': [DATASET],
    'split': [SPLIT],
    'model': [OLLAMA_MODEL],
    'k_neighbors': [K],
    'prompt': [PROMPT],
    'knn': [KNN],
    'temperature': [temperature],
    'top_p': [top_p],
    'num_samples': [len(dt_final)],
    'LLM_Prediction_mean': [y_llm.mean()],
    'GNN_Prediction_mean': [y_gnn.mean()],
    'LLM_Error_mean': [error_llm.mean()],
    'GNN_Error_mean': [error_gnn.mean()],
    'samples_improved': [improved.sum()],
    'samples_worsened': [worsened.sum()],
    'samples_unchanged': [unchanged.sum()],
    'pct_improved': [pct_improved],
    'pct_worsened': [pct_worsened]
})

output_file = 'output/results/llm_gnn_results.csv'
# Append to existing file or create new one with header
if os.path.exists(output_file):
    detailed_results.to_csv(output_file, mode='a', header=False, index=False)
else:
    detailed_results.to_csv(output_file, index=False)
print(f"\n✓ Detailed results row appended to {output_file} [run_id: {run_id}]")

# Save predictions for ROC curve recreation (y_gnn and y_llm as columns)
# Separate CSV for each split, with columns encoding experiment parameters + run_id
predictions_file = f'output/roc_data/predictions_{DATASET}_{SPLIT}.csv'

# Create column names that encode the experiment parameters including run_id
col_suffix = f"{OLLAMA_MODEL}_K{K}_{PROMPT}_KNN{KNN}_temp{temperature}_topp{top_p}_{run_id}".replace(':', '_').replace('.', 'p')
y_gnn_col = f'y_gnn_{col_suffix}'
y_llm_col = f'y_llm_{col_suffix}'

if os.path.exists(predictions_file):
    # Load existing predictions and add new columns
    existing_df = pd.read_csv(predictions_file)
    existing_df[y_gnn_col] = y_gnn
    existing_df[y_llm_col] = y_llm
    existing_df.to_csv(predictions_file, index=False)
    print(f"✓ Added columns {y_gnn_col} and {y_llm_col} to {predictions_file}")
else:
    # Create new file with y_true and first set of predictions
    predictions_df = pd.DataFrame({
        'y_true': y_true,
        y_gnn_col: y_gnn,
        y_llm_col: y_llm
    })
    predictions_df.to_csv(predictions_file, index=False)
    print(f"✓ Created {predictions_file} with columns y_true, {y_gnn_col}, {y_llm_col}")

# Save summary metrics as a single row to shared CSV
summary = {
    'run_id': run_id,
    'dataset': DATASET,
    'split': SPLIT,
    'model': OLLAMA_MODEL,
    'k_neighbors': K,
    'prompt': PROMPT,
    'knn': KNN,
    'temperature': temperature,
    'top_p': top_p,
    'num_samples': len(dt_final),
    'auc_llm': auc_llm,
    'auc_gnn': auc_gnn,
    'auc_improvement': auc_llm - auc_gnn,
    'mse_llm': mse_llm,
    'mse_gnn': mse_gnn,
    'acc_llm': acc_llm,
    'acc_gnn': acc_gnn,
    'f1_llm': f1_llm,
    'f1_gnn': f1_gnn,
    'mcc_llm': mcc_llm,
    'mcc_gnn': mcc_gnn,
    'samples_improved': improved.sum(),
    'samples_worsened': worsened.sum(),
    'samples_unchanged': unchanged.sum(),
    'pct_improved': pct_improved,
    'pct_worsened': pct_worsened,
    'avg_error_reduction': avg_error_reduction
}

summary_df = pd.DataFrame([summary])
summary_file = 'output/summary/llm_gnn_summary.csv'
# Append to existing file or create new one with header
if os.path.exists(summary_file):
    summary_df.to_csv(summary_file, mode='a', header=False, index=False)
else:
    summary_df.to_csv(summary_file, index=False)
print(f"✓ Summary metrics row appended to {summary_file} [run_id: {run_id}]")

print(f"\n{'='*60}")
print("ALL RESULTS SAVED:")
print(f"  • Run ID:       {run_id}")
print(f"  • Results:      {output_file}")
print(f"  • Predictions:  {predictions_file}")
print(f"  • Summary:      {summary_file}")
print(f"{'='*60}")
